# YOLOv11 Keypoint Detection Finetuning

Finetuning YOLOv11, following reference: https://colab.research.google.com/github/roboflow-ai/notebooks/blob/main/notebooks/train-yolov8-keypoint.ipynb

In [1]:
!pip -V

pip 25.3 from /Users/rjunw/Desktop/dev/nba2nba/.venv/lib/python3.11/site-packages/pip (python 3.11)


In [ ]:
# sys deps
import os
from pathlib import Path
from dotenv import load_dotenv

# data deps
from roboflow import Roboflow
from PIL import Image

# model deps
import supervision as sv
from supervision.metrics import MeanAveragePrecision
from ultralytics import YOLO

# tracking deps
import mlflow
from tqdm import tqdm

# env vars
load_dotenv()
ROBOFLOW_API_KEY = os.getenv("ROBOFLOW_API_KEY")

# model vars
LOAD_MODEL = True
YOLOv11x_CHECKPOINT = "../models/weights/yolov11x_kpd/checkpoint_best_total.pth"

# mlflow
mlflow.set_tracking_uri("sqlite:///../mlflow.db")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/Users/rjunw/Library/Application Support/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


## Data Processing

In [4]:
CACHE_DIR = Path("../data/.cache/")
CACHE_DIR.mkdir(parents=True,exist_ok=True)

train_data_path = CACHE_DIR.joinpath("bball_court_kpd/")
print('NBA COURT DETECTION DATA:', train_data_path)

# check if we have files in the directory and use them if we do
if list(train_data_path.glob("*")):
    print(f"Data cached, using {train_data_path}")

# otherwise download from Roboflow
else:
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace("roboflow-jvuqo").project("basketball-court-detection-2")
    version = project.version(19)
    dataset = version.download("yolov8", location=str(train_data_path))

NBA COURT DETECTION DATA: ../data/.cache/bball_court_kpd
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to ../data/.cache/bball_court_kpd in yolov8:: 100%|██████████| 2932/2932 [00:00<00:00, 5612.77it/s]


## NBA Court Keypoint Detection Finetuning

YOLOv11 has it's own formatting to use (not COCO). You can download the Roboflow dataset directly in YOLOv8 format and train, or you can use ultralytics JSON2YOLO to convert COCO format to YOLO.

In [ ]:
train_config = {
    "epochs": 100,
    "imgsz": 640
}

if LOAD_MODEL and Path(YOLOv11x_CHECKPOINT).is_file():
    print("Loading model from checkpoint weights")
    model = YOLO(YOLOv11x_CHECKPOINT)
else:
    mlflow.set_experiment("yolov11_kpd_finetuning")
    print("Finetuning model from COCO pre-trained weights")

    with mlflow.start_run(run_name="yolov11_kpd_finetuning"):
        mlflow.log_params(train_config)
        model = YOLO("yolo11x-pose.pt")
        os.rename("yolo11x-pose.pt", "../models/weights/yolov11x_kpd/yolo11x-pose.pt")
        model.train(
            dataset_dir=str(train_data_path) + "/data.yaml",
            **train_config
        )

    mlflow.end_run()

## Evaluating Fine-Tuned YOLOv11

In [ ]:
mlflow.set_experiment("yolov11_kpd_evaluation")